# Tamper with a record, then forge one that verifies

*Adversarial Mathematics From First Principles*, [chapter 25](https://programmer.ie/books/math/25-chapter/) · [worked example](https://programmer.ie/examples/math/25-chapter/)

Run the cells in order. The program is a small, self-contained demonstration of the chapter's mathematics; the output saved with it is what it printed when this notebook was built.

A run leaves `result.json`, a notebook and a `SHA256SUMS` manifest, and
`math-engine proof --verify` checks every file against it. This program builds
a record and tries four things: verify it fresh, edit the status by hand, drop
in an extra file, and forge the status while rewriting the manifest to match.

## Run it

Standard library only; nothing to install.

In [1]:
"""Records and notebooks: what a hash proves, and what it does not (standard library only)."""
import hashlib
import json
import pathlib
import tempfile


def write_sums(d):
    lines = [f"{hashlib.sha256(p.read_bytes()).hexdigest()}  {p.name}\n"
             for p in sorted(d.iterdir()) if p.name != "SHA256SUMS"]
    (d / "SHA256SUMS").write_text("".join(lines))


def verify(d):
    listed = {}
    for line in (d / "SHA256SUMS").read_text().splitlines():
        h, name = line.split("  ")
        listed[name] = h
    problems = [f"changed: {n}" for n, h in listed.items()
                if not (d / n).is_file() or hashlib.sha256((d / n).read_bytes()).hexdigest() != h]
    problems += [f"untracked: {p.name}" for p in d.iterdir() if p.name not in listed and p.name != "SHA256SUMS"]
    return problems or ["evidence intact"]


with tempfile.TemporaryDirectory() as tmp:
    run = pathlib.Path(tmp)
    result = {"run_id": "20261113T161233Z-demo", "status": "kernel_checked_correspondence_pending"}
    (run / "result.json").write_text(json.dumps(result, indent=2) + "\n")
    (run / "notebook.md").write_text("# Demo\n\nStatus: kernel_checked_correspondence_pending\n")
    write_sums(run)
    print("1. a fresh record:            ", verify(run))

    text = (run / "result.json").read_text().replace("pending", "approved")
    (run / "result.json").write_text(text)
    print("2. status edited by hand:     ", verify(run))

    (run / "result.json").write_text(text.replace("approved", "pending"))
    (run / "extra.log").write_text("added later\n")
    print("3. an extra file dropped in:  ", verify(run))

    (run / "extra.log").unlink()
    (run / "result.json").write_text(text)          # the forged 'approved' status, again
    write_sums(run)                                 # ...and a manifest rewritten to match
    print("4. forged, manifest rewritten:", verify(run))
    print("\nA matching hash says the file is the same file, nothing more. Case 4 verifies:")
    print("the defence against a consistent forgery is independent reproduction, not hashing.")

1. a fresh record:             ['evidence intact']
2. status edited by hand:      ['changed: result.json']
3. an extra file dropped in:   ['untracked: extra.log']
4. forged, manifest rewritten: ['evidence intact']

A matching hash says the file is the same file, nothing more. Case 4 verifies:
the defence against a consistent forgery is independent reproduction, not hashing.


## What the output shows

The edit and the extra file are both caught. The forgery verifies: a
matching hash says the file is the same file as the one the manifest lists,
nothing more. The defence against a consistent forgery is custody (who holds
the record) and reproduction (running it again independently), not hashing.

## Not shown

- The verifier here checks the manifest only; the real `--verify` also checks that each notebook is bound to its `result.json` hash and the contract is unchanged.
- No record here is real; run ids and statuses are made up.

## Challenge it

Examine what `SHA256SUMS` verification can and cannot detect. For each of the
four cases, say what a reader should conclude. Then design a procedure, using
only things outside the record (custody, reproduction, published hashes),
that would detect the forgery in case 4.